# Pandas for Machine Learning — Part 2 (Concepts & Code)

This notebook continues directly from **`Pandas_for_ML_Part_1.ipynb`**. If you haven't completed Part 1 (and its practice notebook), do that first — this notebook assumes you're comfortable with Series, DataFrames, selection, filtering, and handling missing data.

## Learning Objectives
By the end of this notebook, you will be able to:
- Group and aggregate data with `.groupby()` and `.agg()`.
- Combine multiple DataFrames using `.merge()` and `pd.concat()`.
- Build a simple pivot table.
- Apply custom functions to a DataFrame with `.apply()`.
- Use the `.str` accessor to clean and transform text columns.
- Work with dates using `pd.to_datetime()` and basic datetime attributes.
- Detect and remove duplicate rows.
- Rename columns.
- Export a DataFrame back to CSV and JSON.
- Walk through a small, end-to-end sales-data mini analysis.

## Prerequisites
- `Pandas_for_ML_Part_1.ipynb` (and ideally its practice notebook).

## Companion notebook
After this notebook, open **`Pandas_Practice_Part_2.ipynb`** to test yourself. That notebook contains exercises only — no solutions.

---
## Table of Contents
1. [Quick Revision of Part 1](#revision)
2. [GroupBy: Aggregating Data](#groupby)
3. [Multiple Aggregations with `.agg()`](#agg)
4. [Merging DataFrames](#merging)
5. [Concatenating DataFrames](#concatenating)
6. [Pivot Tables](#pivot)
7. [Applying Custom Functions with `.apply()`](#apply)
8. [String Methods with `.str`](#str-accessor)
9. [Working with Dates](#dates)
10. [Duplicates](#duplicates)
11. [Renaming Columns](#renaming)
12. [Exporting Data](#exporting)
13. [Common Mistakes Recap](#common-mistakes)
14. [Mini Project: Sales Data Analysis](#mini-project)
15. [Summary and What's Next](#summary)


<a id="revision"></a>
# 1. Quick Revision of Part 1

Run the cell below to recreate a small dataset and reuse a few Part 1 skills in one shot.

In [ ]:
import pandas as pd
import numpy as np
import os

os.makedirs("sample_data", exist_ok=True)

data = {
    "name": ["Ali", "Sara", "Ahmed", "Ayesha", "Bilal", "Zainab"],
    "department": ["Engineering", "Marketing", "Engineering", "Sales", "Engineering", "Marketing"],
    "city": ["Lahore", "Karachi", "Lahore", "Islamabad", "Karachi", "Islamabad"],
    "salary": [85000, 72000, 120000, 55000, 98000, 68000],
}

df = pd.DataFrame(data)
print(df)
print()
print(df[df["salary"] > 80000])   # Part 1 skill: boolean filtering

<a id="groupby"></a>
# 2. GroupBy: Aggregating Data

`.groupby()` is one of the most powerful tools in Pandas. It follows a "**split → apply → combine**" pattern:
1. **Split** the data into groups based on some column's values.
2. **Apply** an aggregate function (like `mean`, `sum`, `count`) to each group.
3. **Combine** the results back into a single DataFrame or Series.

In [ ]:
grouped = df.groupby("department")
print(grouped["salary"].mean())     # average salary per department

In [ ]:
# A few more common aggregations
print(df.groupby("department")["salary"].sum())
print()
print(df.groupby("department")["salary"].count())
print()
print(df.groupby("department").size())   # number of rows per group (any column)

In [ ]:
# Grouping by more than one column
print(df.groupby(["department", "city"])["salary"].mean())

**Common mistake:** Forgetting that `.groupby()` alone doesn't produce a visible result — it creates a "GroupBy object" waiting for an aggregation. You must chain something like `["salary"].mean()` afterward.

### ML Connection
GroupBy is exactly how you'd compute per-category statistics before modeling — for example, the average house price *per neighborhood*, which might itself become a useful engineered feature.

<a id="agg"></a>
# 3. Multiple Aggregations with `.agg()`

`.agg()` lets you compute **several** aggregate statistics at once, optionally with different functions per column.

In [ ]:
summary = df.groupby("department")["salary"].agg(["mean", "min", "max", "count"])
print(summary)

In [ ]:
# Different aggregations for different columns
summary2 = df.groupby("department").agg(
    average_salary=("salary", "mean"),
    num_employees=("name", "count"),
)
print(summary2)

### ML Connection
This "named aggregation" style (`average_salary=("salary", "mean")`) produces clean, ready-to-use column names — exactly the kind of summary table you might export as a report or feed into further analysis.

<a id="merging"></a>
# 4. Merging DataFrames

Real data often lives in **separate tables** that need to be joined — for example, an "employees" table and a separate "departments" table with extra department-level details. `pd.merge()` joins DataFrames based on a shared key column, similar to a SQL `JOIN`.

In [ ]:
department_info = pd.DataFrame({
    "department": ["Engineering", "Marketing", "Sales"],
    "floor": [3, 5, 5],
    "manager": ["Fatima", "Omar", "Nadia"],
})

print(department_info)

In [ ]:
merged = pd.merge(df, department_info, on="department", how="left")
print(merged)

The `how` parameter controls the join type:

| `how` | Meaning |
|---|---|
| `"left"` | keep all rows from the left DataFrame, matching where possible |
| `"right"` | keep all rows from the right DataFrame |
| `"inner"` | keep only rows where the key exists in **both** DataFrames (default) |
| `"outer"` | keep all rows from **both**, filling gaps with `NaN` |

**Common mistake:** Using the default `how="inner"` when you actually need `"left"` — an inner join silently **drops** rows that don't have a match, which can quietly shrink your dataset without an obvious error.

<a id="concatenating"></a>
# 5. Concatenating DataFrames

`pd.concat()` stacks DataFrames together — typically adding more **rows** (e.g. combining data from two months), though it can also combine columns.

In [ ]:
new_employees = pd.DataFrame({
    "name": ["Hassan", "Mariam"],
    "department": ["Sales", "Engineering"],
    "city": ["Lahore", "Karachi"],
    "salary": [61000, 89000],
})

combined = pd.concat([df, new_employees], ignore_index=True)   # ignore_index resets 0,1,2,... labels
print(combined)

**Common mistake:** Forgetting `ignore_index=True` when concatenating, which leaves duplicate index labels behind (e.g. two different rows both labeled `0`) — this can cause confusing bugs later when using `.loc[]`.

<a id="pivot"></a>
# 6. Pivot Tables

A pivot table reshapes data: one column's unique values become new **columns**, another's become **rows**, and a numeric column is aggregated at each intersection. It's the Pandas equivalent of an Excel pivot table.

In [ ]:
pivot = df.pivot_table(
    values="salary",
    index="department",
    columns="city",
    aggfunc="mean",
)
print(pivot)

Notice the `NaN` values above — they simply mean there's no employee in that specific department/city combination in our small sample dataset.

<a id="apply"></a>
# 7. Applying Custom Functions with `.apply()`

`.apply()` lets you run a custom function across a Series (or DataFrame). Use it when a built-in Pandas/NumPy operation doesn't already do what you need.

In [ ]:
def salary_band(salary):
    if salary >= 100000:
        return "High"
    elif salary >= 70000:
        return "Medium"
    else:
        return "Low"

df["salary_band"] = df["salary"].apply(salary_band)
print(df)

In [ ]:
# A quick lambda (anonymous function) instead of a named function, for simple cases
df["salary_in_thousands"] = df["salary"].apply(lambda s: s / 1000)
print(df[["name", "salary", "salary_in_thousands"]])

**Common mistake:** Reaching for `.apply()` with a Python loop-like function when a plain vectorized operation would work and be much faster (e.g. `df["salary"] / 1000` instead of `.apply(lambda s: s / 1000)`). Use `.apply()` when the logic is genuinely conditional or too complex for direct vectorized arithmetic.

<a id="str-accessor"></a>
# 8. String Methods with `.str`

Just like Python strings have methods (`.upper()`, `.strip()`, etc. — see the Core Python notebooks), Pandas string *columns* have the same methods available through the `.str` accessor, applied to every row at once.

In [ ]:
df["name_upper"] = df["name"].str.upper()
df["city_length"] = df["city"].str.len()

print(df[["name", "name_upper", "city", "city_length"]])

In [ ]:
# Filtering rows based on a string condition
print(df[df["department"].str.startswith("Eng")])

### ML Connection
Cleaning messy text columns — inconsistent capitalization, stray whitespace, unwanted symbols — is one of the most common early steps in preparing real-world data, and `.str` methods are your main tool for it.

<a id="dates"></a>
# 9. Working with Dates

Dates are extremely common in real datasets (order dates, signup dates, timestamps). `pd.to_datetime()` converts text into a proper datetime type, unlocking useful attributes like `.year`, `.month`, and `.day_name()`.

In [ ]:
dates_df = pd.DataFrame({
    "employee": ["Ali", "Sara", "Ahmed"],
    "joining_date": ["2021-03-15", "2022-07-01", "2019-11-30"],
})

dates_df["joining_date"] = pd.to_datetime(dates_df["joining_date"])
print(dates_df.dtypes)
print(dates_df)

In [ ]:
dates_df["joining_year"] = dates_df["joining_date"].dt.year
dates_df["joining_month"] = dates_df["joining_date"].dt.month
dates_df["joining_weekday"] = dates_df["joining_date"].dt.day_name()

print(dates_df)

**Common mistake:** Forgetting to convert a date column with `pd.to_datetime()` first. Without it, the column is just plain text (`dtype: object`), and `.dt` attributes like `.dt.year` will raise an error.

<a id="duplicates"></a>
# 10. Duplicates

In [ ]:
duplicated_df = pd.DataFrame({
    "name": ["Ali", "Sara", "Ali", "Ahmed", "Sara"],
    "department": ["Engineering", "Marketing", "Engineering", "Sales", "Marketing"],
})

print(duplicated_df.duplicated())          # True where a row is an exact repeat of an earlier row
print()
print(duplicated_df.drop_duplicates())     # keeps only the first occurrence of each duplicate

### ML Connection
Duplicate rows can quietly bias a model (by over-representing certain examples) or inflate evaluation scores if the same record appears in both a training and test split. Checking for duplicates is a standard, early data-cleaning step.

<a id="renaming"></a>
# 11. Renaming Columns

In [ ]:
renamed_df = df.rename(columns={"name": "employee_name", "salary": "annual_salary"})
print(renamed_df.columns)
print()
print(df.columns)   # original DataFrame is untouched, as always with non-inplace operations

<a id="exporting"></a>
# 12. Exporting Data

Once you've cleaned or summarized data, you'll often want to save it back to disk — building directly on the serialization ideas from the Core Python notebooks (`json`, `pickle`), Pandas adds convenient, dataset-specific export methods.

In [ ]:
df.to_csv("sample_data/employees_processed.csv", index=False)   # index=False avoids saving the row index as a column
print("Saved to CSV.")

reloaded = pd.read_csv("sample_data/employees_processed.csv")
print(reloaded.head())

In [ ]:
df.to_json("sample_data/employees_processed.json", orient="records", indent=4)
print("Saved to JSON.")

with open("sample_data/employees_processed.json") as file:
    print(file.read()[:300], "...")   # preview the first part of the file

**Common mistake:** Forgetting `index=False` when calling `.to_csv()`. Without it, Pandas writes the DataFrame's row index as an extra, usually unwanted, first column in the file.

### ML Connection
`orient="records"` in `.to_json()` produces exactly the "list of dictionaries" JSON shape you saw in the Core Python serialization section — the same shape that's easy to load back into a DataFrame, or send to another system entirely.

<a id="common-mistakes"></a>
# 13. Common Mistakes — Recap

| Mistake | Fix |
|---|---|
| Calling `.groupby("col")` without an aggregation afterward | Chain an aggregation, e.g. `.groupby("col")["value"].mean()` |
| Using the default `how="inner"` merge when rows might not all match | Choose `how` deliberately (`"left"`, `"right"`, `"outer"`, `"inner"`) |
| Forgetting `ignore_index=True` in `pd.concat()` | Pass it to avoid duplicate index labels |
| Using `.apply()` for simple arithmetic that could be vectorized | Prefer direct vectorized operations when possible, for speed |
| Using `.dt` attributes before converting a column with `pd.to_datetime()` | Always convert first |
| Forgetting `index=False` in `.to_csv()` | Include it to avoid an unwanted extra column |

<a id="mini-project"></a>
# 14. Mini Project: Sales Data Analysis

Let's combine almost everything from both Pandas notebooks into one small, realistic analysis.

In [ ]:
sales_csv_text = """order_id,product,category,region,units_sold,unit_price,order_date
1,Laptop,Electronics,Lahore,3,1200,2024-01-05
2,Mouse,Electronics,Karachi,10,25,2024-01-07
3,Desk,Furniture,Lahore,2,300,2024-01-10
4,Chair,Furniture,Islamabad,5,150,2024-01-12
5,Monitor,Electronics,Karachi,4,220,2024-02-01
6,Laptop,Electronics,Islamabad,2,1200,2024-02-03
7,Desk,Furniture,Karachi,1,300,2024-02-15
8,Chair,Furniture,Lahore,6,150,2024-02-20
9,Monitor,Electronics,Lahore,3,220,2024-03-02
10,Mouse,Electronics,Islamabad,8,25,2024-03-05
"""

with open("sample_data/sales.csv", "w") as file:
    file.write(sales_csv_text)

sales = pd.read_csv("sample_data/sales.csv")
sales["order_date"] = pd.to_datetime(sales["order_date"])
print(sales.head())
print(sales.info())

In [ ]:
# Feature engineering: total revenue per order
sales["revenue"] = sales["units_sold"] * sales["unit_price"]
sales["order_month"] = sales["order_date"].dt.month

print(sales[["order_id", "product", "units_sold", "unit_price", "revenue", "order_month"]])

In [ ]:
# Aggregation: total revenue per category
revenue_by_category = sales.groupby("category")["revenue"].sum().sort_values(ascending=False)
print(revenue_by_category)

In [ ]:
# Aggregation: revenue per region AND category (pivot table)
region_category_pivot = sales.pivot_table(
    values="revenue",
    index="region",
    columns="category",
    aggfunc="sum",
    fill_value=0,     # replace missing combinations with 0 instead of NaN
)
print(region_category_pivot)

In [ ]:
# Which single order generated the most revenue?
top_order = sales.sort_values("revenue", ascending=False).iloc[0]
order_id = top_order["order_id"]
product = top_order["product"]
revenue = top_order["revenue"]
print(f"Highest-revenue order: #{order_id} ({product}) -- ${revenue}")

# Monthly revenue trend
monthly_revenue = sales.groupby("order_month")["revenue"].sum()
print(monthly_revenue)

In [ ]:
# Export a clean summary report
summary_report = sales.groupby(["category", "region"]).agg(
    total_units=("units_sold", "sum"),
    total_revenue=("revenue", "sum"),
).reset_index()

summary_report.to_csv("sample_data/sales_summary.csv", index=False)
print(summary_report)
print("\nSaved summary to sample_data/sales_summary.csv")

### Project Wrap-up

This mini project used almost every Part 2 concept: reading a CSV, converting dates, feature engineering (`revenue`, `order_month`), `groupby` aggregation, pivot tables, sorting, and exporting a cleaned summary — exactly the kind of workflow that precedes feeding data into a Scikit-learn model.

<a id="summary"></a>
# Summary — What You Learned in Part 2

- `.groupby()` and `.agg()` for powerful, flexible aggregation.
- `pd.merge()` for SQL-style joins between related DataFrames.
- `pd.concat()` for stacking DataFrames together.
- Pivot tables for reshaping data into a cross-tab summary.
- `.apply()` for custom row/column transformations.
- The `.str` accessor for cleaning and transforming text columns.
- `pd.to_datetime()` and `.dt` attributes for working with dates.
- Detecting and removing duplicate rows.
- Renaming columns and exporting cleaned data to CSV and JSON.
- A complete mini sales-analysis project tying it all together.

## What's Next?

You now have a solid Pandas foundation for real-world data wrangling. From here, the course continues into **Data Visualization** (seeing the patterns in the data you can now clean and reshape) and then **Scikit-learn**, where the cleaned NumPy/Pandas data you produce becomes the direct input to actual Machine Learning models.

**Next:** Practice everything from this notebook in `Pandas_Practice_Part_2.ipynb`.